In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_iris

np.random.seed(42)

X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

clf = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
clf.fit(X_train, y_train)
preds = clf.predict(X_test)

acc = accuracy_score(y_test, preds)
print(acc)

# Округление вниз до сотых
from math import floor
result = floor(acc * 100) / 100
print(f"{result:.2f}".replace('.', ','))

0.9833333333333333
0,98


In [7]:
import numpy as np

# 1. Загрузка данных из файла PCA.npy
# Предполагается, что файл находится в той же папке, что и скрипт/ноутбук
X = np.load('PCA.npy')

# 2. Вычисление сингулярных чисел с помощью SVD
# compute_uv=False возвращает только сингулярные числа, что немного быстрее
singular_values = np.linalg.svd(X, compute_uv=False)

# 3. Сумма всех сингулярных чисел (полная "энергия" / дисперсия)
total_sum = np.sum(singular_values)

# 4. Поиск минимального m (начиная с 1), такого что E_m < 0.2
# E_m = (сумма сингулярных чисел с индекса m до конца) / total_sum
# Внимание: в задании m - это номер (начинается с 1).
# При m=1 в числителе будут sigma2+...+sigman, что соответствует индексу 1 в массиве Python (второй элемент)
m_found = None
for m in range(1, len(singular_values)):  # m от 1 до n-1 (так как при m=n числитель был бы пуст)
    tail_sum = np.sum(singular_values[m:])  # элементы с индексом m и далее
    Em = tail_sum / total_sum
    if Em < 0.2:
        m_found = m
        break

# Если подходящее m не найдено (например, даже при m=n-1 Em >= 0.2),
# то по логике можно вернуть n (всё пространство), хотя в условии такого нет.
if m_found is None:
    m_found = len(singular_values)

# 5. Вывод результата
print(f"Найденное значение m (номер, начиная с 1): {m_found}")

Найденное значение m (номер, начиная с 1): 12


In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

# Загрузка данных
mnist = fetch_openml('mnist_784')
X = mnist.data.to_numpy()[:2000]
y = mnist.target.to_numpy()[:2000]

# Разбиение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Масштабирование
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Список размерностей
N_COMPONENTS = [1, 3, 5, 10, 15, 20, 30, 40, 50, 60]

best_accuracy = 0
best_n = None

for n in N_COMPONENTS:
    # PCA
    pca = PCA(n_components=n, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    
    # Логистическая регрессия
    lr = LogisticRegression(max_iter=500, random_state=42)
    lr.fit(X_train_pca, y_train)
    
    # Предсказание и accuracy
    y_pred = lr.predict(X_test_pca)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"n_components = {n:2d}, accuracy = {acc:.4f}")
    
    # Выбор лучшего
    if acc > best_accuracy:
        best_accuracy = acc
        best_n = n
    elif acc == best_accuracy and n < best_n:
        best_n = n

print(f"\nОптимальное число компонент: {best_n}")
print(f"Достигнутая accuracy: {best_accuracy:.4f}")